# 🚀 Spark Partitioning & Transformations (Deep Dive)

---

# 1️⃣ What is Partitioning in Spark?

Partition = Small chunk of distributed data.

Spark splits data into partitions so it can:

- Process in parallel
- Distribute across executors
- Improve scalability

---

## 🔹 Key Rule

👉 One partition = One task  
👉 More partitions = More parallelism  

---

## 🔹 Check Number of Partitions

### RDD

```python
rdd = spark.sparkContext.parallelize(range(1, 101), 4)
print(rdd.getNumPartitions())
```

Output:
```
4
```

---

### DataFrame

```python
df = spark.range(0, 100)
print(df.rdd.getNumPartitions())
```

---

# 2️⃣ Why Partitioning Matters

Good partitioning:

- Maximizes CPU usage
- Reduces shuffle
- Prevents skew
- Improves job speed

Bad partitioning:

- Causes long-running tasks
- Creates skew
- Underutilizes cluster

In [0]:
df = spark.range(0, 100)
print(df.rdd.getNumPartitions())

In [0]:
rdd = df.rdd

In [0]:
rdd.collect()
rdd.glom().collect()

# 🚀 Types of Transformations

Spark transformations are of two types:

---

# 🔹 Narrow Transformations
> **A narrow transformation is a transformation where each output partition depends on only one input partition, and therefore no shuffle is required. Ex. filter, select, union, map, etc**
**Note: When ever we create wide transformation Spark create 200 partitions by default.**

No shuffle required.

Each partition depends on only ONE parent partition.

Examples:

- map()
- filter()
- flatMap()
- select()

---

## Example

```python
rdd = spark.sparkContext.parallelize([1,2,3,4], 2)
rdd2 = rdd.map(lambda x: x * 2)
```

Execution:

```
Partition 1 → Partition 1
Partition 2 → Partition 2
```

No data movement.

---

# 🔹 Wide Transformations
> **A wide transformation is a transformation where each output partition depends on multiple input partitions, requiring a shuffle of data across executors. Ex. join, groupby, orderby, distinct , etc**

Shuffle required.

Each partition depends on MULTIPLE parent partitions.

Examples:

- groupBy()
- reduceByKey()
- join()
- distinct()
- orderBy()

---

## Example

```python
rdd = spark.sparkContext.parallelize(
    [("a",1),("b",1),("a",2),("b",3)],
    2
)

rdd2 = rdd.reduceByKey(lambda x,y: x+y)
```

Execution:

```
Partition 1 → Shuffle → New partitions
Partition 2 → Shuffle → New partitions
```

Data redistributed by key.

# 🔥 Hands-On Narrow & Wide Transformation

In [0]:
# Narrow Transformation
# Example 1
# ------------------------------------- *** ------------------------------------------------------------


rdd = spark.sparkContext.parallelize([1,2,3,4], 2)
rdd2 = rdd.map(lambda x: x * 2)
# >> rdd.glom().collect()
# output of rdd = [[1, 2], [3, 4]]

print(rdd2.collect())

In [0]:
rdd.glom().collect()

In [0]:
spark

In [0]:
# Example 2
# ------------------------------------- *** ------------------------------------------------------------

from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

data = [
  ("Alice", 25, "New York"),
  ("Bob", 30, "San Francisco"),
  ("Charlie", 35, "Chicago"),
  ("Arijit", 35, None)
]

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema1 = StructType([

  StructField("name", StringType(), True),
  StructField("age", IntegerType(), True),
  StructField("city", StringType(), True)
])

df = spark.createDataFrame(data, schema1)

%md
# Narrow Transformations

In [0]:
df = df.filter(col("city") == "New York")
df.show()

In [0]:
df.rdd.getNumPartitions()
df.rdd.glom().collect()

%md
# Wide Transformation

In [0]:
rdd = spark.sparkContext.parallelize(
    [("a",1),("b",1),("a",2),("b",3)],
    2
)

rdd2 = rdd.reduceByKey(lambda x,y: x+y)

In [0]:
rdd2.glom().collect()

In [0]:
from operator import add

rdd = spark.sparkContext.parallelize(
    [("apple", 1),
     ("banana", 1),
     ("apple", 1),
     ("banana", 1),
     ("apple", 1)]
)

rdd2 = rdd.reduceByKey(add)
print(rdd2.collect())

In [0]:
def sum_values(x, y):
    return x + y

rdd = spark.sparkContext.parallelize(
    [("a", 10), ("a", 20), ("b", 5)]
)

rdd2 = rdd.reduceByKey(sum_values)
print(rdd2.collect())

In [0]:
# Example 4
# ------------------------------------- *** ------------------------------------------------------------

from pyspark.sql.functions import col, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

data = [
  ("Alice", 25, "New York"),
  ("Bob", 30, "San Francisco"),
  ("Charlie", 35, "Chicago"),
  ("Arijit", 35, None)
]

from pyspark.sql.types import StructType, StructField, StringType, IntegerType
schema1 = StructType([

  StructField("name", StringType(), True),
  StructField("age", IntegerType(), True),
  StructField("city", StringType(), True)
])

df = spark.createDataFrame(data, schema1)

In [0]:
df.head(20)

> **Notes: When calling groupBy().sum() without specifying columns, Spark automatically applies the SUM aggregation to all numeric columns except the grouping column. This behavior is schema-driven and handled by Catalyst optimizer.**

In [0]:
# df.groupBy("city").agg({"age": "avg"}).show()
# df.groupBy("age").sum().show()
df.groupBy("city").sum("age").withColumnRenamed("sum(age)", "total_age").show()

from pyspark.sql.functions import sum, max, col
df.groupBy("city").agg(sum("age").alias("total_age")).show()


df.groupBy("city").agg(max(col("age")).alias("max_age")).show()
df1 = df.groupBy("city").agg({"age": "avg", "name": "count"})

# df.groupBy("city").agg({"age": "avg"}).show()
# df.groupBy("city").agg({"age": "avg", "name": "count"}).show()

In [0]:
df1.rdd.getNumPartitions()
df1.rdd.glom().collect()

In [0]:
df1.show()

In [0]:
df1.explain()

In [0]:
df2 = df.groupBy("city").agg(max(col("age")))

In [0]:
df2.show()

In [0]:
df2.explain()

# 🚀 What is Repartition?

Repartition = Increase or decrease partitions WITH shuffle.

---

## 🔹 Syntax

```python
df2 = df.repartition(10)
```

---

## 🔹 What Happens Internally?

- Full shuffle
- Redistributes data evenly
- Expensive operation

---

## 🔹 When to Use?

- Increase parallelism
- Fix skew
- Before large joins
- Before writing large data

---

## 🔹 Hands-On Example

```python
df = spark.range(0, 100)
print("Before:", df.rdd.getNumPartitions())

df2 = df.repartition(10)

print("After:", df2.rdd.getNumPartitions())
```

---

# 5️⃣ What is Coalesce?

Coalesce = Reduce partitions WITHOUT full shuffle.

---

## 🔹 Syntax

```python
df2 = df.coalesce(2)
```

---

## 🔹 What Happens Internally?

- Avoids full shuffle
- Merges existing partitions
- Less expensive
- Not evenly balanced

---

## 🔹 When to Use?

- Reduce partitions
- Before writing to single file
- Small data consolidation

---

## 🔹 Hands-On Example

```python
df = spark.range(0, 100, 1, 10)
print("Before:", df.rdd.getNumPartitions())

df2 = df.coalesce(2)

print("After:", df2.rdd.getNumPartitions())
```

---

# 6️⃣ Repartition vs Coalesce

| Feature | Repartition | Coalesce |
|----------|-------------|-----------|
| Shuffle | ✅ Yes | ❌ No (by default) |
| Increase partitions | ✅ | ❌ |
| Decrease partitions | ✅ | ✅ |
| Performance cost | High | Low |
| Data balance | Even | May be uneven |

---

# 7️⃣ Practical Real-World Example

---

## Scenario: Writing Large Data

If you write without partition control:

```python
df.write.parquet("/path")
```

If 200 partitions → 200 output files.

To reduce files:

```python
df.coalesce(1).write.parquet("/path")
```

⚠️ Warning: Only safe for small datasets.

---

## Scenario: Improve Parallelism

Small number of partitions:

```python
df = spark.read.parquet("large_file")
```

If only 2 partitions → Underutilized cluster.

Fix:

```python
df = df.repartition(20)
```

---

# 8️⃣ Partitioning by Column

You can partition by key:

```python
df2 = df.repartition("country")
```

Useful for:

- Join optimization
- Reducing shuffle
- Skew control

---

# 9️⃣ How Shuffle Creates Stage Boundary

Wide transformation creates:

New stage in DAG.

Example:

```
map → filter → reduceByKey → map
```

Stages:

Stage 1: map → filter  
Stage 2: reduceByKey  
Stage 3: map  

---

# 🔟 Interview-Level Key Concepts

- One partition = One task
- Narrow transformation → No shuffle
- Wide transformation → Shuffle
- Repartition → Full shuffle
- Coalesce → Partial or no shuffle
- Too many partitions → Task overhead
- Too few partitions → Poor parallelism
- Ideal partition size ≈ 100–200 MB per partition

---

# 1️⃣1️⃣ Advanced Concept: Optimal Partition Size

General rule:

```
Total Data Size / 128MB ≈ Number of Partitions
```

Example:

10 GB data:

```
10,240 MB / 128 MB ≈ 80 partitions
```

---

# 🚀 Summary

Partitioning controls:

- Parallelism
- Performance
- Shuffle behavior
- File output size

Transformations:

- Narrow = Cheap
- Wide = Expensive

Repartition:

- Shuffle
- Balanced
- Expensive

Coalesce:

- No full shuffle
- Fast
- Uneven

---

# 🎯 What You Should Practice

- Check partitions before & after operations
- Use Spark UI to observe shuffle
- Experiment with repartition vs coalesce
- Create skew scenario manually
- Monitor stage boundaries
